In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [0]:
bronze_gtfs_routes = spark.table("bg_traffic.bg_traffic_bronze.gtfs_routes")
bronze_gtfs_routes.display()

In [0]:
required_columns = {
    "route_id",
    "agency_id",
    "route_short_name",
    "route_long_name",
    "route_type",

}

missing_columns = required_columns - set(bronze_gtfs_routes.columns)
if missing_columns:
    raise ValueError(
        "GRESKA: Izvorni GTFS routes je promenio strukturu, postoje nedostajuce kolone!"    
    )

In [0]:
bronze_gtfs_routes.select([F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in bronze_gtfs_routes.columns]).show()

print(f"Total rows: {bronze_gtfs_routes.count()}")

In [0]:
bronze_gtfs_routes = bronze_gtfs_routes.drop('route_url','continuous_pickup','continuous_drop_off','route_text_color','route_color','route_sort_order','route_text_color')
bronze_gtfs_routes.display()

In [0]:
bronze_gtfs_routes.printSchema()

### Casting

In [0]:
typed_routes = bronze_gtfs_routes.select(
    F.col("route_id").cast("integer"),
    F.col("agency_id").cast("string"),
    F.col("route_short_name").cast("string"),
    F.col("route_long_name").cast("string"),
    F.col("route_type").cast("integer"),
)

typed_routes.printSchema()

### Dedup

In [0]:
dedup_routes = typed_routes.dropDuplicates(['route_id'])

dedup_count = dedup_routes.count() - typed_routes.count()
print(f"Broj duplikata: {dedup_count}")

### Valid

In [0]:
valid_routes = dedup_routes.filter(
    (F.col("route_id").isNotNull())
).withColumn("silver_processed_at",F.current_timestamp())
valid_routes.display()

In [0]:
if valid_routes.isEmpty():
    raise Exception("GRESKA: Silver tabela za upisivanje je prazna nakon ciscenja!")

### Write in silver table

In [0]:
valid_routes.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("bg_traffic.bg_traffic_silver.gtfs_route")